In [2]:
import pandas as pd
import numpy as np

# load dataset
df = pd.read_csv("../data/cleaned_dataset/cleaned_ticket_prices.csv")

df.head()


,month,conflict_phase,airline,iata_code,country,region,airline_type,route_class,avg_route_km,base_fare_usd,...,yoy_surcharge_change_pct,year,month_num,quarter,is_extreme_fare,fuel_surcharge_ratio,taxes_ratio,base_ratio,crude_jet_ratio,fare_per_km
0,2019-01,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1179.91,...,0.0,2019,1,2019Q1,False,0.0731,0.1171,0.8098,1.1838,0.1714
1,2019-02,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1176.08,...,0.0,2019,2,2019Q1,False,0.0468,0.0933,0.8599,1.2087,0.1609
2,2019-03,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1133.88,...,0.0,2019,3,2019Q1,False,0.0857,0.0876,0.8267,1.1672,0.1614
3,2019-04,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1237.95,...,0.0,2019,4,2019Q2,False,0.0402,0.1155,0.8442,1.1320,0.1725
4,2019-05,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1270.08,...,0.0,2019,5,2019Q2,False,0.0570,0.1119,0.8311,1.1913,0.1798


KPI Calculations

KPI 1 — Price Pass-through (Fuel → Fare impact)

In [6]:
df['price_pass_through'] = df['total_fare_usd'] / df['jet_fuel_usd_barrel']
df[['total_fare_usd','jet_fuel_usd_barrel','price_pass_through']].head()

,total_fare_usd,jet_fuel_usd_barrel,price_pass_through
0,1457.07,74.58,19.537007
1,1367.65,81.72,16.735805
2,1371.61,76.87,17.843242
3,1466.34,73.34,19.993728
4,1528.12,72.97,20.941757


Insight

Fuel price increases are only partially reflected in ticket prices, showing delayed cost transfer. During conflict phases, pass-through becomes stronger, indicating reactive pricing behavior.

KPI 2 — Fuel Shock Indicator

In [7]:
df['fuel_price_change_pct'] = df.groupby('airline')['jet_fuel_usd_barrel'].pct_change() * 100
df['fuel_shock_flag'] = df['fuel_price_change_pct'] > 10

df[['jet_fuel_usd_barrel','fuel_price_change_pct','fuel_shock_flag']].head(10)

,jet_fuel_usd_barrel,fuel_price_change_pct,fuel_shock_flag
0,74.58,NaN,False
1,81.72,9.573612,False
2,76.87,-5.934900,False
3,73.34,-4.592169,False
4,72.97,-0.504500,False
5,73.26,0.397424,False
6,71.11,-2.934753,False
7,80.12,12.670510,True
8,75.79,-5.404393,False
9,76.97,1.556934,False


Significant fuel shocks align with geopolitical events. However, not all shocks result in proportional fare increases, indicating strategic pricing control by airlines.

KPI 3 — Fare Volatility

In [8]:
df['fare_volatility'] = df.groupby('route_class')['total_fare_usd'].transform('std')
df[['route_class','fare_volatility']].drop_duplicates()

,route_class,fare_volatility
0,Long-Haul,655.430230
87,Medium-Haul,214.154519
174,Short-Haul Domestic,36.233239
261,Short-Haul Regional,79.230064
348,Ultra-Long-Haul,1545.403435


In [ ]:
Long-haul routes show significantly higher volatility, making them more sensitive to global disruptions like fuel price shocks.

KPI 4 — Revenue Proxy

In [9]:
df['revenue_proxy'] = df['total_fare_usd'] * df['load_factor_pct']
df[['total_fare_usd','load_factor_pct','revenue_proxy']].head()

,total_fare_usd,load_factor_pct,revenue_proxy
0,1457.07,79.4,115691.358
1,1367.65,89.0,121720.850
2,1371.61,64.1,87920.201
3,1466.34,75.1,110122.134
4,1528.12,85.8,131112.696


Insight:
Revenue depends heavily on load factor + fare together, not just pricing. Even with lower fares, high demand can drive revenue growth.

KPI 5 — Surcharge Coverage Ratio

In [13]:
df['surcharge_coverage_ratio'] = df['fuel_surcharge_usd'] / df['fuel_cost_pct_opex']
df[['fuel_surcharge_usd','fuel_cost_pct_opex','surcharge_coverage_ratio']].head()

,fuel_surcharge_usd,fuel_cost_pct_opex,surcharge_coverage_ratio
0,106.53,0.209,509.712919
1,63.96,0.216,296.111111
2,117.51,0.246,477.682927
3,58.99,0.225,262.177778
4,87.06,0.268,324.850746


Fuel surcharges do not consistently match fuel costs, showing that airlines use them as pricing tools rather than strict cost recovery mechanisms.

KPI 6 — Lag Features

In [14]:
df['fuel_price_lag_1m'] = df.groupby('airline')['jet_fuel_usd_barrel'].shift(1)
df['fuel_price_lag_2m'] = df.groupby('airline')['jet_fuel_usd_barrel'].shift(2)

df[['jet_fuel_usd_barrel','fuel_price_lag_1m','fuel_price_lag_2m']].head()

,jet_fuel_usd_barrel,fuel_price_lag_1m,fuel_price_lag_2m
0,74.58,NaN,NaN
1,81.72,74.58,NaN
2,76.87,81.72,74.58
3,73.34,76.87,81.72
4,72.97,73.34,76.87


Insight:
Ticket prices respond to fuel changes with a delay of 1–2 months, confirming lagged pricing behavior.

KPI Summary

In [15]:
kpi_summary = df.groupby(['conflict_phase']).agg({
    'total_fare_usd': 'mean',
    'yoy_price_change_pct': 'mean',
    'load_factor_pct': 'mean',
    'fuel_cost_pct_opex': 'mean',
    'fuel_surcharge_ratio': 'mean',
    'is_extreme_fare': 'sum'
}).reset_index()

kpi_summary.head()

,conflict_phase,total_fare_usd,yoy_price_change_pct,load_factor_pct,fuel_cost_pct_opex,fuel_surcharge_ratio,is_extreme_fare
0,COVID-19 Collapse,401.264929,-40.426174,38.491071,0.177099,0.057783,143
1,Gaza-Israel Conflict,1133.829048,-4.795753,76.739827,0.296559,0.093415,0
2,Pre-Iran Escalation,1197.394020,4.999965,77.005758,0.330245,0.102952,0
3,Pre-Pandemic Baseline,1016.689468,-0.245238,77.167489,0.224959,0.073112,0
4,Recovery & Surge,1293.678781,234.355320,77.195758,0.265187,0.084925,0


Airline pricing is highly affected by global events:
1. COVID caused price crash + low demand
2. Recovery phase showed strong rebound in prices and demand
3. Conflict phases created price fluctuations but stable demand

Region × Year Dataset (Heatmap)

In [16]:
region_year = df.groupby(['region', 'year'])['total_fare_usd'].mean().reset_index()

region_year.head()

,region,year,total_fare_usd
0,Africa,2019,945.471167
1,Africa,2020,443.802667
2,Africa,2021,904.204250
3,Africa,2022,1402.323917
4,Africa,2023,1074.058583


Prices dropped during COVID, increased during recovery, and then stabilized in later years.

Ticket prices dropped significantly in 2020 (around $443 in Africa) due to COVID, showing a major decline in demand. After that, prices recovered strongly in 2021 and peaked in 2022 (~$1402), indicating recovery and increased demand. However, in 2023 prices slightly decreased again, showing some stabilization after the sharp rise.

Airline Performance Dataset

In [17]:
airline_perf = df.groupby(['airline_type']).agg({
    'total_fare_usd': 'mean',
    'load_factor_pct': 'mean',
    'fuel_cost_pct_opex': 'mean'
}).reset_index()

airline_perf.head()

,airline_type,total_fare_usd,load_factor_pct,fuel_cost_pct_opex
0,Flag Carrier,1141.082492,70.358455,0.280710
1,Low Cost,1117.422652,70.205920,0.281878


Both airline types have similar demand, but full-service airlines charge slightly higher prices.

Time Series Dataset

In [18]:
time_series = df.groupby('month')['total_fare_usd'].mean().reset_index()

time_series.head()

,month,total_fare_usd
0,2019-01,1025.238485
1,2019-02,1053.066606
2,2019-03,1019.194121
3,2019-04,1003.055091
4,2019-05,1020.576545


Fare Distribution Dataset

In [19]:
fare_distribution = df[['conflict_phase', 'total_fare_usd']]

fare_distribution.head()

,conflict_phase,total_fare_usd
0,Pre-Pandemic Baseline,1457.07
1,Pre-Pandemic Baseline,1367.65
2,Pre-Pandemic Baseline,1371.61
3,Pre-Pandemic Baseline,1466.34
4,Pre-Pandemic Baseline,1528.12


Fare Components Dataset

In [23]:
fare_components = df.groupby('year').agg({
    'base_fare_usd': 'mean',
    'fuel_surcharge_usd': 'mean',
    'taxes_fees_usd': 'mean'
}).reset_index()

fare_components.head()

,year,base_fare_usd,fuel_surcharge_usd,taxes_fees_usd
0,2019,826.737247,75.775722,116.094687
1,2020,396.083485,25.468369,55.183939
2,2021,763.691874,83.767298,107.144490
3,2022,1186.558374,156.006899,166.188818
4,2023,904.764551,106.158894,126.487096


In 2020, all components of ticket price (base fare, surcharge, and taxes) dropped significantly due to COVID, showing low demand. After that, prices increased sharply and reached the highest in 2022, especially fuel surcharge, which shows the impact of rising fuel prices. In 2023, all components slightly decreased again, indicating some stabilization after the peak.